# EYEWAZ — train your Urdu Piper voice (self-contained)

Every fix baked in. **Run all** top to bottom.

**Setup once:** Runtime → Change runtime type → **GPU (T4)**. In Google Drive,
make a folder `eyewaz` and put **`dataset-male.zip`** inside it. That's it.

⚠️ **Keep this browser tab open and click into it now and then** — Colab kills
idle sessions and wipes everything. Your dataset lives safely in Drive, so if it
does reset, just **Run all** again.


### 1. Mount Drive (dataset lives here, survives resets)


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os; DRIVE='/content/drive/MyDrive/eyewaz'; os.makedirs(DRIVE, exist_ok=True)
VOICE='male'
assert os.path.exists(f'{DRIVE}/dataset-{VOICE}.zip'), f'Put dataset-{VOICE}.zip in Google Drive/eyewaz/ first'
print('dataset found in Drive ✓')


### 2. GPU on?


In [ ]:
!nvidia-smi -L || print('NO GPU — Runtime > Change runtime type > GPU first!')


### 3. Install piper1-gpl (compiles espeakbridge, keeps the vits trainer)
~4–5 min. Does the non-editable build to compile the native bridge, then an
editable install for the training source, then stitches them together.


In [ ]:
import os, glob, shutil
!apt-get -qq install -y espeak-ng libespeak-ng-dev build-essential cmake ninja-build >/dev/null
if not os.path.exists('/content/piper1'):
    !git clone -q https://github.com/OHF-Voice/piper1-gpl /content/piper1
os.chdir('/content/piper1')
# 1) non-editable build -> compiles espeakbridge*.so into site-packages
!pip install -q . 2>&1 | tail -4
so = glob.glob('/usr/local/lib/python3.12/dist-packages/piper/espeakbridge*.so')
assert so, 'espeakbridge did not compile — paste !ls /usr/local/lib/python3.12/dist-packages/piper'
shutil.copy(so[0], '/content/eb.so')   # back it up before the editable reinstall removes it
# 2) editable install -> exposes piper.train.vits from source
!pip install -q -e . --force-reinstall --no-deps
# 3) drop the compiled bridge into the editable source package
shutil.copy('/content/eb.so', '/content/piper1/src/piper/'+os.path.basename(so[0]))
# 4) build the Cython alignment module
!bash build_monotonic_align.sh
!python3 -c "from piper import espeakbridge; from piper.train.vits.dataset import VitsDataModule; print('imports OK ✓')"


### 4. Dataset + pretrained checkpoint


In [ ]:
import os
os.chdir('/content'); VOICE='male'; DRIVE='/content/drive/MyDrive/eyewaz'
!cp {DRIVE}/dataset-{VOICE}.zip /content/
!unzip -oq /content/dataset-{VOICE}.zip -d /content/
with open(f'/content/dataset-{VOICE}/metadata.csv',encoding='utf-8') as f, open(f'/content/{VOICE}.csv','w',encoding='utf-8') as o:
    for line in f:
        line=line.strip()
        if '|' in line:
            cid,text=line.split('|',1); o.write(f'{cid}.wav|{text}\n')
print('clips:', len(os.listdir(f'/content/dataset-{VOICE}/wav')))
# pretrained checkpoint to fine-tune from (cached in Drive so resets don't re-download)
if not os.path.exists(f'{DRIVE}/pretrained.ckpt'):
    !wget -q -O {DRIVE}/pretrained.ckpt 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt'
print('pretrained MB:', round(os.path.getsize(f'{DRIVE}/pretrained.ckpt')/1e6,1))


### 5. Train
**Watch the live `Epoch N` progress with loss values.** Leave it running 30+ min
(tab open). Checkpoints save to `/content/out`. Stop with ■ when you've trained
enough, then run cell 6.


In [ ]:
import torch, os, glob
VOICE='male'; DRIVE='/content/drive/MyDrive/eyewaz'
latest = sorted(glob.glob('/content/out/**/*.ckpt', recursive=True), key=os.path.getmtime)
if latest:
    RESUME = latest[-1]; print('resuming from', RESUME)
else:
    ck = torch.load(f'{DRIVE}/pretrained.ckpt', map_location='cpu', weights_only=False)
    for k in ['hyper_parameters','hparams_name','datamodule_hyper_parameters','loops']: ck.pop(k, None)
    ck['epoch']=0; ck['global_step']=0
    for s in (ck.get('lr_schedulers') or []):
        if isinstance(s, dict): s['last_epoch']=0; s['_step_count']=1
    torch.save(ck,'/content/warm.ckpt'); RESUME='/content/warm.ckpt'; print('warm-start from Lessac')
os.chdir('/content/piper1')
open('/content/piper1/sitecustomize.py','w').write("import torch\n_o=torch.load\ndef _p(*a,**k):\n k['weights_only']=False\n return _o(*a,**k)\ntorch.load=_p\n")
!PYTHONPATH=/content/piper1 python3 -m piper.train fit \
  --data.voice_name eyewaz-urdu-{VOICE} \
  --data.csv_path /content/{VOICE}.csv \
  --data.audio_dir /content/dataset-{VOICE}/wav \
  --model.sample_rate 22050 \
  --data.espeak_voice ur \
  --data.cache_dir /content/cache \
  --data.config_path {DRIVE}/eyewaz-urdu-{VOICE}.json \
  --data.batch_size 16 \
  --ckpt_path {RESUME} \
  --trainer.max_epochs 1000 \
  --trainer.default_root_dir /content/out \
  --trainer.accelerator gpu --trainer.devices 1


### 6. Export to ONNX (saved to Drive + downloaded)


In [ ]:
import glob, os
VOICE='male'; DRIVE='/content/drive/MyDrive/eyewaz'
cks = sorted(glob.glob('/content/out/**/*.ckpt', recursive=True), key=os.path.getmtime)
assert cks, 'No checkpoint yet — let cell 5 run more epochs.'
ck = cks[-1]; print('exporting', ck)
!python3 -m piper.train.export_onnx --checkpoint {ck} --output-file {DRIVE}/eyewaz-urdu-{VOICE}.onnx
!cp {DRIVE}/eyewaz-urdu-{VOICE}.json {DRIVE}/eyewaz-urdu-{VOICE}.onnx.json
from google.colab import files
files.download(f'{DRIVE}/eyewaz-urdu-{VOICE}.onnx'); files.download(f'{DRIVE}/eyewaz-urdu-{VOICE}.onnx.json')
print('also saved in Drive/eyewaz/')


### Done
The voice files are in **Drive/eyewaz/** (survive resets) and downloaded. Send
me `eyewaz-urdu-male.onnx` + `.onnx.json` and I wire it into every EYEWAZ surface.
